# 02 — Trees and Binary Search Trees

## Why This Matters for DSA
Linear structures like arrays, vectors, and linked lists are inherently limited: search is either $O(\log n)$ (but only if sorted, and insertion/deletion is $O(n)$ to shift elements) or $O(n)$ (if unsorted or a linked list). Trees, specifically Binary Search Trees (BSTs), break this linear barrier by introducing hierarchical structure. On average, they allow search, insertion, and deletion to all execute in $O(\log n)$ time. This notebook bridges the gap between basic pointer structures (`07_Linked_Lists`) and more advanced hierarchical concepts like self-balancing trees (`04_Self_Balancing_and_Tries`) and heaps (`01_Heaps_and_Priority_Queues`). Trees are also the natural home for both Depth-First Search (DFS) and Breadth-First Search (BFS), techniques you will see generalized to graphs later.

## Prerequisites
- `02_Recursion` (Phase 01) — tree traversals and operations are fundamentally recursive; you must be comfortable with recursive calls and base cases
- `07_Linked_Lists` (Phase 02) — dynamic tree storage is just a linked structure with two pointers instead of one
- `08_Stacks_Queues_Deques` (Phase 02) — queues are used for BFS level-order traversal, and stacks are used for converting postfix expressions to expression trees

## Learning Objectives
By the end of this notebook, you will be able to:
- Define and apply rooted tree terminology: root, leaf, internal node, sibling, degree, path, depth, and height.
- Distinguish between Full, Perfect, Almost Complete, and Balanced binary trees, and state their mathematical properties.
- Represent binary trees using both contiguous arrays (index-1 mapping) and linked pointer-based nodes (`TreeNode`).
- Implement preorder, inorder, postorder, and level-order (BFS) traversals on binary trees.
- Build and evaluate expression trees from postfix expressions using a stack.
- Explain the BST invariant and implement search, insertion, and deletion (handling all 3 cases with inorder successor) using C++ reference-to-pointer parameters.
- Analyze BST performance, identify tree degeneracy (worst-case $O(n)$ behavior), and explain how insertion order determines height.
- Judge when a BST is the appropriate structure versus when a hash map or sorted array fits a given workload.

## Checklist Before Moving On
- [ ] I can define path length, depth, and height, and calculate them by hand on any tree
- [ ] I can explain why a perfect binary tree of height $h$ has $2^h$ leaves and $2^{h+1}-1$ total nodes
- [ ] I can write the bitwise index calculations for array-based tree storage (parent, left child, right child)
- [ ] I can write recursive preorder, inorder, and postorder traversals without referencing code
- [ ] I can build an expression tree from postfix and explain why postorder traversal evaluates it
- [ ] I can write BST search, insertion, and deletion (specifically Case 3: 2 children) from memory
- [ ] I can explain what a degenerate tree is and why it reduces BST operations to $O(n)$ time

## Table of Contents
1. Tree Terminology and Classifications — Full, Perfect, Almost Complete, and Balanced
2. Binary Tree Storage — Contiguous Array vs. Linked Nodes
3. Tree Traversals — Preorder, Inorder, Postorder, and BFS
4. Expression Trees — Construction and Recursive Evaluation
5. BST Fundamentals — Search and the Binary Search Invariant
6. BST Insertion and Deletion — Reference-to-Pointer Technique
7. BST Degeneracy and Performance Analysis
8. When to Use (and Not Use) a Tree
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. Tree Terminology and Classifications — Full, Perfect, Almost Complete, and Balanced

### The Why
A rooted tree is a hierarchy of nodes connected by directed edges. Unlike linear structures, a tree can branch in multiple directions, creating child subtrees. Mastering the terminology and mathematical classifications of trees is critical because properties like depth, height, and completeness directly determine the runtime complexity of operations on the tree.

### Core Mechanism
- **Nodes & Relationships:**
  - **Root:** The top node in the tree; the only node with no parent.
  - **Parent & Child:** If a node points to another, the pointing node is the parent and the pointed-to node is the child.
  - **Sibling:** Nodes that share the same parent.
  - **Leaf Node:** A node with zero children (degree 0).
  - **Internal Node:** A node with one or more children (degree > 0).
  - **Degree:** The number of children a node has.
- **Paths & Dimensions:**
  - **Path:** A sequence of nodes $(a_0, a_1, \dots, a_n)$ where $a_{i+1}$ is a child of $a_i$. The **length** of a path is the number of edges, which is $\text{number of nodes} - 1$.
  - **Depth (or Level):** The length of the unique path from the root to a node. The root has depth 0.
  - **Height:** The maximum depth of any node in the tree. The height of a tree with a single node is 0. An empty tree has height -1.
  - **Ancestors & Descendants:** If a path exists from node $A$ to node $B$, $A$ is an ancestor of $B$, and $B$ is a descendant of $A$. Strict ancestors/descendants exclude the node itself.
  - **Subtree:** A portion of a tree that is itself a tree, rooted at some child node.
- **Binary Tree Classifications:**
  - **Binary Tree:** A tree where every node has at most two children, labeled as `left` and `right`.
  - **Full Node:** A node with both left and right subtrees non-empty.
  - **Full (Proper/Strict) Binary Tree:** A binary tree where every node has either 0 or 2 children. No node has exactly 1 child.
  - **Perfect (or Complete) Binary Tree:** A binary tree of height $h$ where all leaves are at depth $h$, and all internal nodes are full.
    - Number of leaf nodes: $L = 2^h$.
    - Number of internal nodes: $2^h - 1$.
    - Total number of nodes: $n = 2^{h+1} - 1 = 2L - 1$.
  - **Almost (or Nearly) Complete Binary Tree:** A binary tree of height $h$ where:
    1. Every level $d < h$ is completely filled (having $2^d$ nodes).
    2. At level $h$, all nodes are filled from left to right without gaps.
    - This is the exact shape used in binary heaps (`01_Heaps_and_Priority_Queues`).
    - Total nodes $n$ satisfies $2^h \le n < 2^{h+1}$.
    - Height $h = \lfloor \log_2 n \rfloor$.
  - **Balanced Binary Tree:** A binary tree where the difference in height between the left and right subtrees of *every* node is at most 1.
  - **Completely Balanced Binary Tree:** A binary tree where the left and right subtrees of *every* node have the exact same height.

### Common Pitfalls
- **Height of a single node:** Confusing height and node count. The height of a single-node tree is **0** (no edges), not 1. An empty tree has height **-1**.
- **Complete vs. Almost Complete:** Assuming "complete" in computer science literature always means "almost complete". Some textbooks call a perfect tree "complete", and a heap-structured tree "almost complete". Be careful with definitions: look at the exact constraints (no gaps at the bottom level, filled left to right).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <cmath>
using namespace std;

// Calculations for Binary Tree Properties
// Based on the formulas from Chapter 15 - Trees
int main() {
    int h = 3; // Height of tree
    
    // For a PERFECT Binary Tree of height h:
    int leaves = pow(2, h);
    int internal_nodes = leaves - 1;
    int total_nodes = pow(2, h + 1) - 1;
    
    cout << "=== Perfect Binary Tree of Height " << h << " ===\n";
    cout << "Number of Leaf Nodes (2^h):      " << leaves << "\n";
    cout << "Number of Internal Nodes (2^h-1): " << internal_nodes << "\n";
    cout << "Total Number of Nodes (2^(h+1)-1): " << total_nodes << "\n\n";
    
    // For an ALMOST COMPLETE Binary Tree of n nodes:
    int n = 12; // Example node count
    int height = log2(n); // h = floor(log2(n))
    
    cout << "=== Almost Complete Binary Tree with " << n << " nodes ===\n";
    cout << "Calculated Height (floor(log2(n))): " << height << "\n";
    cout << "Min possible nodes for height " << height << ": " << (1 << height) << "\n";
    cout << "Max possible nodes for height " << height << ": " << ((1 << (height + 1)) - 1) << "\n";
    
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Binary Tree Storage — Contiguous Array vs. Linked Nodes

### The Why
A binary tree can be represented in memory using two fundamentally different layouts: contiguous array storage or pointer-based linked node storage. Understanding the trade-offs is essential: contiguous storage is optimal for heaps because it requires zero pointer overhead, but it is extremely inefficient for unbalanced or sparse trees. Linked nodes provide maximum flexibility for insertion and restructuring at the cost of memory overhead for pointers.

### Core Mechanism
- **Contiguous Array Storage:**
  - Traverse the tree in breadth-first order (level-by-level, left-to-right) and place elements in an array.
  - To simplify the mathematical formulas, start storing elements at **index 1** instead of index 0.
  - For a node at index $k$:
    - **Left Child Index:** $2k$ (implement as `k << 1`)
    - **Right Child Index:** $2k + 1$ (implement as `(k << 1) | 1`)
    - **Parent Index:** $\lfloor k / 2 \rfloor$ (implement as `k >> 1`)
  - **The Disadvantage:** If the tree is sparse or unbalanced (skewed), many array positions will remain unused. A skewed tree of height $h$ with only $h+1$ nodes requires an array size of $2^{h+1}$, wasting $O(2^h)$ memory space.
- **Linked Node Storage:**
  - Each node is represented dynamically as an object containing:
    1. An information/value field (`value` or `info`).
    2. A pointer to the left child node (`left` or `leftChild`).
    3. A pointer to the right child node (`right` or `rightChild`).
  - An external root pointer points to the root node (analogous to the `head` pointer in a linked list).

### Common Pitfalls
- **Off-by-one with Bitwise Operations:** Using bitwise shifts on 0-indexed arrays. If the array starts at 0, the left child of node $k$ is at $2k+1$, and the right child is at $2k+2$. This is slightly more complex than starting at index 1, where left is $2k$ and right is $2k+1$.
- **Wasted Memory in Arrays:** Choosing array storage for arbitrary binary trees without considering their balance. A tree with nodes only along the rightmost branches will quickly overflow normal array allocations even for a small number of elements.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
using namespace std;

// 1. Contiguous Array-Based Storage of a Binary Tree
class ArrayBinaryTree {
    vector<string> tree;
    string sentinel = "NULL"; // placeholder for empty positions
public:
    ArrayBinaryTree(int cap) : tree(cap, sentinel) {}

    void setNode(int index, string val) {
        if (index < (int)tree.size()) tree[index] = val;
    }

    void printRelations(int k) {
        if (k >= (int)tree.size() || tree[k] == sentinel) {
            cout << "No node at index " << k << "\n";
            return;
        }
        cout << "Node [" << tree[k] << "] at index " << k << ":\n";
        
        int parent = k >> 1; // k / 2
        int left = k << 1;   // 2 * k
        int right = (k << 1) | 1; // 2 * k + 1

        if (parent >= 1 && tree[parent] != sentinel)
            cout << "  Parent:      [" << tree[parent] << "] at index " << parent << "\n";
        else
            cout << "  Parent:      None (Root)\n";
        
        if (left < (int)tree.size() && tree[left] != sentinel)
            cout << "  Left Child:  [" << tree[left] << "] at index " << left << "\n";
        else
            cout << "  Left Child:  None\n";
        
        if (right < (int)tree.size() && tree[right] != sentinel)
            cout << "  Right Child: [" << tree[right] << "] at index " << right << "\n";
        else
            cout << "  Right Child: None\n";
    }
};

// 2. Linked Pointer-Based Storage of a Binary Tree
struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

int main() {
    // --- CONTIGUOUS STORAGE DEMONSTRATION ---
    cout << "=== Array-Based Storage (Index starts at 1) ===\n";
    // Build tree: Root A (1), Left B (2), Right C (3), B's Left D (4), B's Right E (5)
    ArrayBinaryTree arrTree(10);
    arrTree.setNode(1, "A");
    arrTree.setNode(2, "B");
    arrTree.setNode(3, "C");
    arrTree.setNode(4, "D");
    arrTree.setNode(5, "E");

    arrTree.printRelations(2); // Check relations for 'B'
    arrTree.printRelations(1); // Check relations for 'A' (Root)

    // --- LINKED STORAGE DEMONSTRATION ---
    cout << "\n=== Linked Pointer-Based Storage ===\n";
    TreeNode* root = new TreeNode(10);
    root->left = new TreeNode(5);
    root->right = new TreeNode(15);
    
    cout << "Root value:       " << root->value << "\n";
    cout << "Left child value: " << root->left->value << "\n";
    cout << "Right child val:  " << root->right->value << "\n";

    // Clean up
    delete root->left;
    delete root->right;
    delete root;

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Tree Traversals — Preorder, Inorder, Postorder, and BFS

### The Why
Unlike linear structures that have a single natural traversal direction, trees can be walked in multiple ways. We divide traversals into **Depth-First Search (DFS)**—which goes deep along a path before backtracking—and **Breadth-First Search (BFS)**—which visits nodes level-by-level. Traversals are the basic API for all tree-based algorithms. For example, Inorder traversal visits BST nodes in sorted order, while Postorder is essential for deleting or evaluating tree structures.

### Core Mechanism
- **Depth-First Traversals (DFS):**
  For each node, we have three steps: visiting the node itself (V), traversing the left subtree (L), and traversing the right subtree (R). This leads to three standard traversal orders:
  1. **Preorder (V-L-R):** Visit current node first, then recurse left, then recurse right.
     - Use case: Copying trees, printing directory hierarchies.
  2. **Inorder (L-V-R):** Recurse left first, then visit current node, then recurse right.
     - Use case: Printing BST values in sorted order.
  3. **Postorder (L-R-V):** Recurse left first, then recurse right, then visit current node.
     - Use case: Deleting the tree (must delete children before deleting parent), expression evaluation.
- **Breadth-First Traversal (BFS / Level-Order):**
  Explore level-by-level from top to bottom, left to right.
  - **Implementation:** Uses a queue (`std::queue`). Push the root. While queue is not empty: dequeue front node, visit it, enqueue its left child, then enqueue its right child.
  - **Connection to BFS:** This is the exact same BFS structure introduced in `08_Stacks_Queues_Deques` Section 5.

### Common Pitfalls
- **Null Pointer Dereference:** Failing to check if the current node pointer is `nullptr` before traversing its children in recursive functions.
- **BFS Enqueueing Nulls:** Pushing null children onto the BFS queue. Check `node->left != nullptr` before pushing.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <queue>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// PREORDER: Visit, Left, Right
void preorder(TreeNode *p) {
    if (p != nullptr) {
        cout << p->value << " ";
        preorder(p->left);
        preorder(p->right);
    }
}

// INORDER: Left, Visit, Right
void inorder(TreeNode *p) {
    if (p != nullptr) {
        inorder(p->left);
        cout << p->value << " ";
        inorder(p->right);
    }
}

// POSTORDER: Left, Right, Visit
void postorder(TreeNode *p) {
    if (p != nullptr) {
        postorder(p->left);
        postorder(p->right);
        cout << p->value << " ";
    }
}

// BREADTH-FIRST / LEVEL ORDER (using a queue)
void levelOrder(TreeNode *root) {
    if (root == nullptr) return;
    queue<TreeNode*> q;
    q.push(root);
    
    while (!q.empty()) {
        TreeNode *cur = q.front();
        q.pop();
        cout << cur->value << " ";
        
        if (cur->left != nullptr)  q.push(cur->left);
        if (cur->right != nullptr) q.push(cur->right);
    }
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Constructing the tree:
    //         1
    //       /   \
    //      2     3
    //     / \   /
    //    4   5 6
    TreeNode* root = new TreeNode(1);
    root->left = new TreeNode(2);
    root->right = new TreeNode(3);
    root->left->left = new TreeNode(4);
    root->left->right = new TreeNode(5);
    root->right->left = new TreeNode(6);

    cout << "Preorder Traversal:  "; preorder(root);   cout << "\n"; // Expected: 1 2 4 5 3 6
    cout << "Inorder Traversal:   "; inorder(root);    cout << "\n"; // Expected: 4 2 5 1 6 3
    cout << "Postorder Traversal: "; postorder(root);  cout << "\n"; // Expected: 4 5 2 6 3 1
    cout << "Level-Order (BFS):   "; levelOrder(root); cout << "\n"; // Expected: 1 2 3 4 5 6

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Expression Trees — Construction and Recursive Evaluation

### The Why
Every mathematical expression has an inherent tree-like hierarchy (precedence of operators). In an **Expression Tree**, leaf nodes are operands (constants/variables) and internal nodes are operators. The tree structure naturally captures order of operations, rendering parentheses unnecessary. Compilers construct expression trees during parsing to evaluate expressions or generate machine code.

### Core Mechanism
- **Construction from Postfix Expression:**
  Using a stack of `TreeNode*`:
  1. Scan the postfix tokens from left to right.
  2. If the token is an **operand**: Create a new leaf node and push its pointer onto the stack.
  3. If the token is an **operator**: Create a new node. Pop the top node from the stack and make it the **right** child of the operator node; pop the next node and make it the **left** child. Push the operator node back onto the stack.
  4. At the end of the expression, the stack contains exactly one node, which is the root of the expression tree.
- **Recursive Evaluation:**
  Evaluate the expression tree using a **Postorder Traversal (L-R-V)**:
  - Base case: If the node is a leaf, return its operand value.
  - Recursive case: Evaluate the left subtree, evaluate the right subtree, then apply the operator at the current node to the two values.

### Common Pitfalls
- **Incorrect Operand Order:** When popping child nodes for an operator, the first popped node is the **right** child, and the second popped node is the **left** child. Reversing this will break subtraction (`-`) and division (`/`).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <stack>
#include <string>
#include <vector>
using namespace std;

struct TreeNode {
    string value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(string val) : value(val), left(nullptr), right(nullptr) {}
};

// Recursive Evaluation (Postorder traversal)
int evaluate(TreeNode *root) {
    if (root == nullptr) return 0;
    
    // Leaf node: return value of operand
    if (root->left == nullptr && root->right == nullptr) {
        return stoi(root->value);
    }
    
    // Operator node: evaluate subtrees
    int leftVal = evaluate(root->left);
    int rightVal = evaluate(root->right);
    
    if (root->value == "+") return leftVal + rightVal;
    if (root->value == "-") return leftVal - rightVal;
    if (root->value == "*") return leftVal * rightVal;
    if (root->value == "/") return leftVal / rightVal;
    
    return 0;
}

// Build Expression Tree from Postfix expression
TreeNode* buildExpressionTree(const vector<string>& tokens) {
    stack<TreeNode*> st;
    for (const string& token : tokens) {
        if (token == "+" || token == "-" || token == "*" || token == "/") {
            TreeNode *node = new TreeNode(token);
            // FIRST pop is the RIGHT child
            node->right = st.top(); st.pop();
            // SECOND pop is the LEFT child
            node->left = st.top(); st.pop();
            st.push(node);
        } else {
            st.push(new TreeNode(token));
        }
    }
    return st.top();
}

void printInOrder(TreeNode *root) {
    if (root) {
        if (root->left || root->right) cout << "(";
        printInOrder(root->left);
        cout << " " << root->value << " ";
        printInOrder(root->right);
        if (root->left || root->right) cout << ")";
    }
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // infix equivalent: (1 + 2) * (3 + 4) = 21
    // postfix representation: 1 2 + 3 4 + *
    vector<string> postfix = {"1", "2", "+", "3", "4", "+", "*"};
    
    TreeNode* root = buildExpressionTree(postfix);
    
    cout << "Expression reconstructed in infix: ";
    printInOrder(root);
    cout << "\n";
    
    cout << "Evaluation Result:                  " << evaluate(root) << "\n"; // Expected: 21

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. BST Fundamentals — Search and the Binary Search Invariant

### The Why
A Binary Search Tree (BST) is a binary tree with a key property: for every node, all values in its left subtree are strictly less than the node's value, and all values in its right subtree are strictly greater. This **BST invariant** organizes the keys so that we can discard half the remaining search space with each comparison—mirroring binary search (`03_Hashing_and_Binary_Search`) but on a dynamic linked structure.

### Core Mechanism
- **The Invariant:**
  $$\forall \text{ node } N, \quad x \in \text{LeftSubtree}(N) \Rightarrow x < N.\text{value} \quad \text{and} \quad y \in \text{RightSubtree}(N) \Rightarrow y > N.\text{value}$$
)- **Searching/Finding a Node:**
  Start at the root. Compare search value `num` to `node->value`:
  - If equal, the node is found. Return `true`.
  - If `num < node->value`, move to the left child.
  - If `num > node->value`, move to the right child.
  - If you hit `nullptr`, the value does not exist in the tree. Return `false`.
  - **Iterative vs. Recursive:** Searching is naturally tail-recursive, which means we can implement it as a simple `while` loop, avoiding stack frame overhead.

### Common Pitfalls
- **Assuming Duplicates are Handled Automatically:** By default, standard BST definitions do not allow duplicate keys. When inserting or searching, duplicate values must either be explicitly rejected or handled by a specific rule (e.g., storing them consistently in the left/right subtree or keeping a frequency count in each node).
- **Infinite Loops:** Forgetting to advance the search pointer inside the traversal loop, leading to an infinite cycle.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

// Iterative Search: O(h) time, O(1) space
bool findIterative(TreeNode* root, int num) {
    TreeNode* cur = root;
    while (cur != nullptr) {
        if (cur->value == num) return true;
        else if (num < cur->value) cur = cur->left;
        else cur = cur->right;
    }
    return false;
}

// Recursive Search: O(h) time, O(h) stack space
bool findRecursive(TreeNode* root, int num) {
    if (root == nullptr) return false;
    if (root->value == num) return true;
    if (num < root->value) return findRecursive(root->left, num);
    return findRecursive(root->right, num);
}

int main() {
    // Construct BST:
    //        10
    //       /  \
    //      5    15
    //     /
    //    3
    TreeNode* root = new TreeNode(10);
    root->left = new TreeNode(5);
    root->right = new TreeNode(15);
    root->left->left = new TreeNode(3);

    cout << "Search 15 (Iterative): " << findIterative(root, 15) << "\n"; // Expected: 1
    cout << "Search 7 (Iterative):  " << findIterative(root, 7) << "\n";  // Expected: 0
    cout << "Search 3 (Recursive):  " << findRecursive(root, 3) << "\n";  // Expected: 1
    
    // Cleanup
    delete root->left->left;
    delete root->left;
    delete root->right;
    delete root;

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. BST Insertion and Deletion — Reference-to-Pointer Technique

### The Why
Modifying a linked tree structure requires modifying node pointers. Without clean tracking, modifying parent pointers (like updating the root or a child connection) requires complex parent-pointer tracking. The **reference-to-pointer (`TreeNode *&`)** technique simplifies this by allowing us to pass the actual pointer alias into functions. Modifying the parameter modifies the parent's child link directly, eliminating structural asymmetry.

### Core Mechanism
- **BST Insertion:**
  Insertions are always performed at empty leaf locations.
  - Step through the tree starting from the root.
  - If the value matches an existing key, reject it (no duplicates).
  - Otherwise, when you reach a null child pointer, attach the new node there.
- **BST Deletion:**
  Deleting a node is more complex because we must re-establish the BST invariant.
  - **Case 1: Leaf Node.** Simple deletion. Free node memory and set the parent's child pointer to `nullptr`.
  - **Case 2: One Child.** Promote the single child subtree. Point the parent's link directly to the child node, then delete the current node.
  - **Case 3: Two Children.** We cannot simply remove the node because it has two disjoint subtrees.
    - **Better Solution:** Find the **minimum value in the right subtree** (the inorder successor).
    - Replace the target node's value with the successor's value.
    - Recursively call `deleteNode` to remove the successor's value from the right subtree. Because the successor is the minimum in the right subtree, it is guaranteed to have at most one child (its left child is null), reducing deletion to Case 1 or Case 2.
- **Reference-to-Pointer:**
  Passing a pointer as a reference (`TreeNode *&nodePtr`) means that when we execute `nodePtr = nodePtr->left`, we are modifying the *actual* left/right link variable inside the parent node (or the root pointer itself).

### Common Pitfalls
- **Memory Leaks:** Forgetting to call `delete` on the removed node pointer.
- **Dangling Pointers:** Failing to set the parent's pointer to `nullptr` after deleting a leaf node, causing subsequent lookups to dereference deleted memory.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

class IntBinaryTree {
private:
    TreeNode *root;

    void destroySubTree(TreeNode *node) {
        if (node) {
            destroySubTree(node->left);
            destroySubTree(node->right);
            delete node;
        }
    }

    // Deletion implementation using reference-to-pointer (TreeNode *&)
    void deleteNode(int num, TreeNode *&nodePtr) {
        if (nodePtr == nullptr) {
            cout << num << " not found.\n";
        } else if (num < nodePtr->value) {
            deleteNode(num, nodePtr->left);  // nodePtr->left is passed by reference
        } else if (num > nodePtr->value) {
            deleteNode(num, nodePtr->right); // nodePtr->right is passed by reference
        } else {
            makeDeletion(nodePtr); // Found the node, perform deletion
        }
    }

    void makeDeletion(TreeNode *&nodePtr) {
        TreeNode *tempNodePtr = nullptr;
        if (nodePtr == nullptr) {
            return;
        } 
        // Case 1 & 2: Leaf node or node with only one child
        else if (nodePtr->right == nullptr) {
            tempNodePtr = nodePtr;
            nodePtr = nodePtr->left; // Reattaches the left child directly
            delete tempNodePtr;
        } else if (nodePtr->left == nullptr) {
            tempNodePtr = nodePtr;
            nodePtr = nodePtr->right; // Reattaches the right child directly
            delete tempNodePtr;
        } 
        // Case 3: Node with two children
        else {
            // Find minimum node (inorder successor) in right subtree
            TreeNode *successor = nodePtr->right;
            while (successor->left != nullptr) {
                successor = successor->left;
            }
            // Replace value with successor value
            nodePtr->value = successor->value;
            // Recursively delete the successor node from the right subtree
            deleteNode(successor->value, nodePtr->right);
        }
    }

    void displayInOrder(TreeNode *node) {
        if (node) {
            displayInOrder(node->left);
            cout << node->value << " ";
            displayInOrder(node->right);
        }
    }

public:
    IntBinaryTree() : root(nullptr) {}
    ~IntBinaryTree() { destroySubTree(root); }

    // Insert node iteratively
    void insertNode(int num) {
        TreeNode *newNode = new TreeNode(num);
        if (!root) {
            root = newNode;
        } else {
            TreeNode *nodePtr = root;
            while (nodePtr) {
                if (num < nodePtr->value) {
                    if (nodePtr->left == nullptr) {
                        nodePtr->left = newNode;
                        break;
                    }
                    nodePtr = nodePtr->left;
                } else if (num > nodePtr->value) {
                    if (nodePtr->right == nullptr) {
                        nodePtr->right = newNode;
                        break;
                    }
                    nodePtr = nodePtr->right;
                } else {
                    cout << "Duplicate value " << num << " not inserted.\n";
                    delete newNode;
                    break;
                }
            }
        }
    }

    void remove(int num) {
        deleteNode(num, root);
    }

    void showInOrder() {
        displayInOrder(root);
        cout << "\n";
    }
};

int main() {
    IntBinaryTree tree;
    
    // Building BST
    tree.insertNode(38);
    tree.insertNode(18);
    tree.insertNode(5);
    tree.insertNode(21);
    tree.insertNode(81);
    tree.insertNode(75);
    tree.insertNode(99);

    cout << "Original tree (Inorder): ";
    tree.showInOrder(); // Expected: 5 18 21 38 75 81 99

    cout << "Removing leaf node 21:\n";
    tree.remove(21);
    tree.showInOrder(); // Expected: 5 18 38 75 81 99

    cout << "Removing node 18 (one child):\n";
    tree.remove(18);
    tree.showInOrder(); // Expected: 5 38 75 81 99

    cout << "Removing node 81 (two children):\n";
    tree.remove(81);
    tree.showInOrder(); // Expected: 5 38 75 99

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. BST Degeneracy and Performance Analysis

### The Why
The main reason we use a BST is to achieve logarithmic $O(\log n)$ runtime for search, insertion, and deletion. However, this guarantee depends entirely on the tree being reasonably balanced. If values are inserted in a skewed order (like sorted values), the tree collapses into a linear structure, destroying the search performance.

### Core Mechanism
- **Insertion Order Determines Shape:**
  - If we insert the values `[1, 2, 3, 4, 5]` in order, each element becomes the right child of the previous one. This creates a **degenerate tree** (or skewed tree), which behaves exactly like a singly linked list.
  - If we insert the same values in order `[3, 1, 5, 2, 4]`, we get a perfectly balanced tree of height 2.
- **Time Complexity Analysis:**
  - **Balanced Tree:** The height is $O(\log n)$. All operations search, insert, and delete visit at most one path from root to leaf, executing in $O(\log n)$ time.
  - **Degenerate Tree:** The height is $O(n)$. Operations require traversing the full depth of the linear chain, executing in $O(n)$ time.
  - **Average Case:** Random insertions yield a height of $O(\log n)$ on average.
  - **The Fix:** Self-balancing binary trees (like AVL trees, `04_Self_Balancing_and_Tries`) enforce balance invariants by rotating subtrees during insertions and deletions, guaranteeing $O(\log n)$ worst-case performance.

| Operation | Average Case (Balanced) | Worst Case (Degenerate) |
|---|---|---|
| Search | $O(\log n)$ | $O(n)$ |
| Insert | $O(\log n)$ | $O(n)$ |
| Delete | $O(\log n)$ | $O(n)$ |
| Space | $O(n)$ | $O(n)$ |

### Common Pitfalls
- **Trusting a standard BST for sorted inputs:** Building a BST from sorted databases or logs without balancing. It will run in $O(n^2)$ total construction time ($n$ insertions at $O(n)$ each) instead of the expected $O(n \log n)$.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

TreeNode* insert(TreeNode* root, int val) {
    if (root == nullptr) return new TreeNode(val);
    if (val < root->value) root->left = insert(root->left, val);
    else if (val > root->value) root->right = insert(root->right, val);
    return root;
}

int countSearchSteps(TreeNode* root, int target) {
    int steps = 0;
    TreeNode* cur = root;
    while (cur != nullptr) {
        steps++;
        if (cur->value == target) return steps;
        else if (target < cur->value) cur = cur->left;
        else cur = cur->right;
    }
    return steps; // Not found
}

int getHeight(TreeNode* root) {
    if (root == nullptr) return -1;
    return 1 + max(getHeight(root->left), getHeight(root->right));
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    vector<int> sortedVals = {1, 2, 3, 4, 5, 6, 7};
    vector<int> balancedVals = {4, 2, 6, 1, 3, 5, 7};

    TreeNode* degenerateRoot = nullptr;
    for (int x : sortedVals) {
        degenerateRoot = insert(degenerateRoot, x);
    }

    TreeNode* balancedRoot = nullptr;
    for (int x : balancedVals) {
        balancedRoot = insert(balancedRoot, x);
    }

    cout << "=== TREE HEIGHT COMPARISON ===\n";
    cout << "Degenerate Tree Height: " << getHeight(degenerateRoot) << " (nodes: 7)\n"; // Expected: 6
    cout << "Balanced Tree Height:   " << getHeight(balancedRoot) << " (nodes: 7)\n\n";   // Expected: 2

    cout << "=== SEARCH STEPS FOR VALUE 7 ===\n";
    cout << "Steps in Degenerate Tree: " << countSearchSteps(degenerateRoot, 7) << "\n"; // Expected: 7
    cout << "Steps in Balanced Tree:   " << countSearchSteps(balancedRoot, 7) << "\n";   // Expected: 3

    cleanTree(degenerateRoot);
    cleanTree(balancedRoot);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. When to Use (and Not Use) a Tree

### The Why
Trees are not always the best tool. We must understand when to use them over alternatives like vectors or hash maps. A hash map gives $O(1)$ search but lacks order; a sorted vector gives $O(\log n)$ search but $O(n)$ insertion. A BST offers a balance—giving $O(\log n)$ for *both* search and insertion, while keeping keys sorted.

### Core Comparison
- **Use a BST when:**
  - You need to perform lookups, insertions, and deletions in a dynamic set, *and* you need to maintain the elements in sorted order.
  - You need to execute range queries (e.g., find all keys between $X$ and $Y$).
  - You need to easily find the minimum, maximum, predecessor, or successor.
- **Do NOT use a BST (or general trees) when:**
  - You only need rapid key-value lookups without ordering requirements. A Hash Map (unordered_map, `03_Hashing_and_Binary_Search`) is faster—$O(1)$ average-case search and insert.
  - The dataset is static. A sorted vector with binary search offers faster read times, better cache locality, and zero pointer memory overhead.
  - Random access by index is required. Trees do not support $O(1)$ random indexing.

| Feature | Sorted Vector | Hash Map | Balanced BST |
|---|---|---|---|
| Search | $O(\log n)$ | $O(1)$ | $O(\log n)$ |
| Insert | $O(n)$ | $O(1)$ | $O(\log n)$ |
| Delete | $O(n)$ | $O(1)$ | $O(\log n)$ |
| Min / Max | $O(1)$ | $O(n)$ | $O(\log n)$ (or $O(h)$) |
| Order Traversals | $O(n)$ | $O(n \log n)$ (must sort) | $O(n)$ (Inorder) |
| Cache Locality | Excellent | Poor | Poor |


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Tree Properties Cheat Sheet

| Property | Perfect Tree (Height $h$) | Skewed Tree (Nodes $n$) | Balanced Tree (Nodes $n$) |
|---|---|---|---|
| Height | $h$ | $n-1$ | $O(\log n)$ |
| Total Nodes | $2^{h+1}-1$ | $n$ | $n$ |
| Leaves Count | $2^h$ | $1$ | $\approx n/2$ |
| Max Depth | $h$ | $n-1$ | $O(\log n)$ |

### Checkpoint Questions
1. If a binary tree has height 4, what is the maximum possible number of leaves it can have? What is the minimum possible?
2. Explain the parent-child index equations in a 1-indexed contiguous array tree representation.
3. How does Inorder traversal behave differently on a general Binary Tree vs. a Binary Search Tree (BST)?
4. Why is Postorder traversal the only natural choice for deleting a dynamic tree?
5. During BST deletion, why does Case 3 (two children) require replacing the deleted node with the minimum of the right subtree?
6. Why is a hash map generally preferred over a BST if we only need key-value lookups?

### Answer Key
1. The maximum possible number of leaves is for a perfect tree of height 4, which is $2^4 = 16$. The minimum possible number of leaves is 1 (for a skewed/degenerate linear tree of height 4).
2. For a node at index $k$, its parent is at index $k \gg 1$, its left child is at index $k \ll 1$, and its right child is at index $(k \ll 1) | 1$.
3. On a general Binary Tree, Inorder traversal visits nodes in structural sequence (left-root-right) without numerical order. On a BST, the Inorder traversal is guaranteed to visit nodes in ascending sorted order.
4. Postorder visits the left child and right child *before* visiting the node itself. This allows us to safely delete the children first and free their memory before freeing the parent node, avoiding dangling pointer dereferences.
5. Replacing the node with the minimum of the right subtree (inorder successor) preserves the BST invariant: all elements in the left subtree remain smaller than it, and all elements in the right subtree remain larger. Additionally, this successor node is guaranteed to have at most one child, making its removal straightforward.
6. A hash map operates in $O(1)$ average time for search, insertion, and deletion, whereas a balanced BST operates in $O(\log n)$ time. The BST is only preferred when we also require sorted order or range queries.

### Mini Project
Implement a function to find the **Lowest Common Ancestor (LCA)** of two nodes in a BST.
- **Rules of BST LCA:** Start from the root. If both target values are smaller than the current node's value, the LCA must be in the left subtree. If both target values are larger, the LCA must be in the right subtree. If one is smaller and one is larger (or one equals the current node's value), the current node is the LCA.


In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode *left;
    TreeNode *right;
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {}
};

TreeNode* insert(TreeNode* root, int val) {
    if (root == nullptr) return new TreeNode(val);
    if (val < root->value) root->left = insert(root->left, val);
    else if (val > root->value) root->right = insert(root->right, val);
    return root;
}

// Find Lowest Common Ancestor (LCA) in a BST
// Time Complexity: O(h), Space Complexity: O(1) iterative
TreeNode* lowestCommonAncestor(TreeNode* root, int p, int q) {
    TreeNode* cur = root;
    while (cur != nullptr) {
        // If both values are smaller, LCA must be in the left subtree
        if (p < cur->value && q < cur->value) {
            cur = cur->left;
        }
        // If both values are larger, LCA must be in the right subtree
        else if (p > cur->value && q > cur->value) {
            cur = cur->right;
        }
        // Split point: one is smaller, one is larger -> this is the LCA
        else {
            return cur;
        }
    }
    return nullptr;
}

void cleanTree(TreeNode *root) {
    if (root) {
        cleanTree(root->left);
        cleanTree(root->right);
        delete root;
    }
}

int main() {
    // Construct BST:
    //        20
    //       /  \
    //      8    22
    //     / \
    //    4   12
    //       /  \
    //      10  14
    TreeNode* root = nullptr;
    root = insert(root, 20);
    root = insert(root, 8);
    root = insert(root, 22);
    root = insert(root, 4);
    root = insert(root, 12);
    root = insert(root, 10);
    root = insert(root, 14);

    TreeNode* lca1 = lowestCommonAncestor(root, 4, 14);
    cout << "LCA of 4 and 14:  " << (lca1 ? lca1->value : -1) << "\n"; // Expected: 8

    TreeNode* lca2 = lowestCommonAncestor(root, 10, 22);
    cout << "LCA of 10 and 22: " << (lca2 ? lca2->value : -1) << "\n"; // Expected: 20

    cleanTree(root);
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. LeetCode Practice Problems

Practice is essential to gain confidence with tree traversals and BST properties. Use these problems to test your understanding.

### Tree Traversal Patterns

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 144 | [Binary Tree Preorder Traversal](https://leetcode.com/problems/binary-tree-preorder-traversal/) | Easy | Implement recursively (V-L-R) and iteratively using a stack |
| 94 | [Binary Tree Inorder Traversal](https://leetcode.com/problems/binary-tree-inorder-traversal/) | Easy | Implement recursively (L-V-R) and iteratively |
| 145 | [Binary Tree Postorder Traversal](https://leetcode.com/problems/binary-tree-postorder-traversal/) | Easy | Implement recursively (L-R-V) — children must be visited first |
| 102 | [Binary Tree Level Order Traversal](https://leetcode.com/problems/binary-tree-level-order-traversal/) | Medium | BFS using a queue, tracking size of each level |

### Binary Search Tree Operations

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 700 | [Search in a Binary Search Tree](https://leetcode.com/problems/search-in-a-binary-search-tree/) | Easy | Section 5, directly — use comparison rules to decide whether to walk left or right |
| 701 | [Insert into a Binary Search Tree](https://leetcode.com/problems/insert-into-a-binary-search-tree/) | Medium | Insert at the appropriate leaf position, maintaining invariant |
| 450 | [Delete Node in a BST](https://leetcode.com/problems/delete-node-in-a-bst/) | Medium | Section 6, directly — handle cases: leaf, 1 child, and 2 children with successor |
| 98 | [Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/) | Medium | Pass valid ranges (min, max) down recursively or use Inorder traversal and verify it is sorted |

### Tree Properties & Ancestors

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 104 | [Maximum Depth of Binary Tree](https://leetcode.com/problems/maximum-depth-of-binary-tree/) | Easy | Height = $1 + \max(\text{depth}(left), \text{depth}(right))$ recursively |
| 110 | [Balanced Binary Tree](https://leetcode.com/problems/balanced-binary-tree/) | Easy | Calculate height of children, verify difference $\le 1$ at every node |
| 235 | [Lowest Common Ancestor of a Binary Search Tree](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/) | Medium | Section 9's Mini Project — find the split point in BST values |
| 222 | [Count Complete Tree Nodes](https://leetcode.com/problems/count-complete-tree-nodes/) | Easy/Medium | Can do recursively in $O(n)$, or in $O(\log^2 n)$ by comparing left-most and right-most heights |

### Self-Check Before Moving to `03_Tree_DFS_BFS`
If you can explain why a perfect binary tree of height $h$ has $2^h$ leaf nodes, write recursive traversals from scratch, and implement Case 3 of BST deletion using the inorder successor, you have mastered the essentials of this notebook. In `03_Tree_DFS_BFS`, we will build directly on these fundamentals to solve complex tree search patterns, path findings, and tree backtracking.
